
# Titanic - Machine Learning na prática

Este notebook foi feito para estudo prático de Machine Learning usando a base do Titanic.

## Objetivo
Prever a variável `Survived` a partir de atributos dos passageiros.

## Variáveis usadas
- `Pclass`
- `Sex`
- `Age`
- `SibSp`
- `Parch`
- `Fare`
- `Embarked`

## Fluxo
1. Ler os dados
2. Tratar valores ausentes
3. Transformar variáveis categóricas
4. Separar treino e validação
5. Treinar o modelo
6. Avaliar resultado
7. Gerar predições para o `test.csv`


## Modelo deste notebook
**K-Nearest Neighbors (KNN)**

In [2]:

import os
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

if not os.path.exists("datas") and os.path.exists("../datas"):
    os.chdir("..")

train_df = pd.read_csv("datas/train.csv")
test_df = pd.read_csv("datas/test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()


Train shape: (891, 12)
Test shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
target = 'Survived'

display(train_df[features + [target]].isnull().sum().to_frame('missing_values'))

fig = px.histogram(train_df, x='Sex', color='Survived', barmode='group',
                   title='Sobrevivência por sexo')
fig.show()

fig = px.histogram(train_df, x='Pclass', color='Survived', barmode='group',
                   title='Sobrevivência por classe')
fig.show()


,missing_values
Pclass,0
Sex,0
Age,177
SibSp,0
Parch,0
Fare,0
Embarked,2
Survived,0


## Pré-processamento

In [4]:

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
target = 'Survived'

X = train_df[features].copy()
y = train_df[target].copy()
X_test_final = test_df[features].copy()

# Tratamento de valores ausentes
for col in ['Age', 'Fare']:
    median_value = X[col].median()
    X[col] = X[col].fillna(median_value)
    X_test_final[col] = X_test_final[col].fillna(median_value)

mode_embarked = X['Embarked'].mode()[0]
X['Embarked'] = X['Embarked'].fillna(mode_embarked)
X_test_final['Embarked'] = X_test_final['Embarked'].fillna(mode_embarked)

# Codificação categórica
X = pd.get_dummies(X, columns=['Sex', 'Embarked'], drop_first=True)
X_test_final = pd.get_dummies(X_test_final, columns=['Sex', 'Embarked'], drop_first=True)

# Garantir mesmas colunas entre treino e teste
X, X_test_final = X.align(X_test_final, join='left', axis=1, fill_value=0)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
X_train.head()


X_train: (712, 8)
X_valid: (179, 8)


,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
692,3,28.0,0,0,56.4958,True,False,True
481,2,28.0,0,0,0.0000,True,False,True
527,1,28.0,0,0,221.7792,True,False,True
855,3,18.0,0,1,9.3500,False,False,True
801,2,31.0,1,1,26.2500,False,False,True



## Escalonamento + KNN

O KNN compara distâncias entre pontos.
Por isso, ele funciona melhor quando as variáveis estão em escala semelhante.


In [5]:

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test_final)

model = KNeighborsClassifier(n_neighbors=7)
model.fit(X_train_scaled, y_train)


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",7
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


## Avaliação e geração da submissão

In [6]:

valid_pred = model.predict(X_valid_scaled)
valid_acc = accuracy_score(y_valid, valid_pred)

print("Acurácia:", round(valid_acc, 4))
print("\nRelatório de classificação:")
print(classification_report(y_valid, valid_pred))

cm = confusion_matrix(y_valid, valid_pred)
cm_df = pd.DataFrame(cm, index=['Real_0', 'Real_1'], columns=['Pred_0', 'Pred_1'])
display(cm_df)

test_pred = model.predict(X_test_scaled)
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': test_pred
})
submission.to_csv('submission_knn.csv', index=False)
print("Arquivo salvo: submission_knn.csv")
submission.head()


Acurácia: 0.8212

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.84      0.87      0.86       110
           1       0.78      0.74      0.76        69

    accuracy                           0.82       179
   macro avg       0.81      0.81      0.81       179
weighted avg       0.82      0.82      0.82       179



,Pred_0,Pred_1
Real_0,96,14
Real_1,18,51


Arquivo salvo: submission_knn.csv


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0



## Teste extra com vários valores de k

Como o KNN depende do número de vizinhos, vale a pena testar diferentes valores.


In [7]:

results = []
for k in range(3, 16, 2):
    temp_model = KNeighborsClassifier(n_neighbors=k)
    temp_model.fit(X_train_scaled, y_train)
    pred = temp_model.predict(X_valid_scaled)
    acc = accuracy_score(y_valid, pred)
    results.append({'k': k, 'accuracy': acc})

results_df = pd.DataFrame(results)
results_df


,k,accuracy
0,3,0.810056
1,5,0.815642
2,7,0.821229
3,9,0.804469
4,11,0.804469
5,13,0.810056
6,15,0.821229
